# Skill 36: Variational Autoencoder (VAE) Mathematics

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Derive the evidence lower bound and implement the reparameterisation trick with a reconstruction + KL loss.

## Objectives
- Produce latent mean and log-variance vectors.
- Sample z with $z=\mu+\sigma\varepsilon$ while preserving gradients.
- Calculate KL divergence from the approximate posterior to a standard normal prior.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
$$z=\mu+\sigma\odot\varepsilon,\quad \mathrm{KL}=-\tfrac12\sum\big(1+\ell-\mu^2-e^{\ell}\big),\quad \mathcal L=\mathcal L_{\text{recon}}+\mathrm{KL}$$
with $\ell=\log\sigma^2$.

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VAE(nn.Module):
    def __init__(self, input_dim=784, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU())
        self.mu = nn.Linear(256, latent_dim)
        self.logvar = nn.Linear(256, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, input_dim), nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.mu(h), self.logvar(h)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

def vae_loss(recon, x, mu, logvar):
    recon_loss = F.binary_cross_entropy(recon, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl, recon_loss, kl

model = VAE()
x = torch.rand(8, 784)
recon, mu, logvar = model(x)
loss, rec, kl = vae_loss(recon, x, mu, logvar)
loss.backward()
print(recon.shape, float(loss), float(rec), float(kl))


## What to check
**Expected result:** A forward/backward pass completes with finite losses (data is random and unseeded, so values vary; recon is about 4,300–4,400 for 8x784 inputs at initialisation).

## Pitfalls
BCE needs inputs in [0,1]. Watch for posterior collapse. At test time use $z=\mu$ for reconstruction.